In [2]:
from datasets import load_dataset

human_eval = load_dataset('openai/openai_humaneval')

/Users/sauravprateek/Documents/saurav-codes/neural-nets-codelab/saurav-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
passed = 0
total = 0

for id, evaluation in enumerate(human_eval['test']):
    code = evaluation['prompt'] + '\n' + evaluation['canonical_solution']
    code_with_tests = code + evaluation['test']
    exec(code_with_tests)
    try:
        check(globals()[evaluation['entry_point']])
        passed += 1
    except Exception as error:
        print(f'Error while executing row: {id} | {error}')
        print(code_with_tests)
    finally:
        total += 1

print(f'Tests passed {passed} / {total}')
    

Tests passed 164 / 164


In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
import os
import json
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

SYSTEM_PROMPT = """
You are a coding assistant whose job is to complete the provided method implementation in Python and return the complete executable python method.
"""

class GeneratedCode(BaseModel):
    code: str = Field(description="Generated code solution")

def generate_code_solution(prompt):
    client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
    response = client.models.generate_content(
        model="gemini-3.1-pro-preview",
        contents=f"{SYSTEM_PROMPT}\n\n{prompt}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=GeneratedCode
        )
    )
    generated_code = json.loads(response.text)
    return generated_code['code'].strip()

In [23]:
# Model Evaluation #
passed = 0
total = 0

for id, evaluation in enumerate(human_eval['test']):
    try:
        solution = generate_code_solution(evaluation['prompt'])
        code_with_tests = solution + '\n' + evaluation['test']
        exec(code_with_tests)
        check(globals()[evaluation['entry_point']])
        passed += 1
    except SyntaxError as e:
        print(f'Syntax Error while executing row: {id} | {e}')
        print(code_with_tests)
    except Exception as error:
        print(f'Error while executing row: {id} | {error}')
        print(code_with_tests)
    finally:
        total += 1
        print(f'[RUNNING] Tests passing: {passed} / {total}')

print(f'[COMPLETED] Tests passed: {passed} / {total}')
    

[RUNNING] Tests passing: 1 / 1
[RUNNING] Tests passing: 2 / 2
[RUNNING] Tests passing: 3 / 3
[RUNNING] Tests passing: 4 / 4
[RUNNING] Tests passing: 5 / 5
[RUNNING] Tests passing: 6 / 6
[RUNNING] Tests passing: 7 / 7
[RUNNING] Tests passing: 8 / 8
[RUNNING] Tests passing: 9 / 9
[RUNNING] Tests passing: 10 / 10
[RUNNING] Tests passing: 11 / 11
Syntax Error while executing row: 11 | unmatched ')' (<string>, line 11)
from typing import List


def string_xor(a: str, b: str) -> str:
    """ Input are two strings a and b consisting only of 1s and 0s.
    Perform binary XOR on these inputs and return result also as a string.
    >>> string_xor('010', '110')
    '100'
    """
    return "".join("0" if x == y else "1" for x, y in zip(a, b))
))


METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candidate('111000', '101010') == '010010'
    assert candidate('1', '1') == '0'
    assert candidate('0101', '0000') == '0101'

[RUNNING] Tests passing: 11 / 12
